# SDF boundary parameterization comparison

This notebook is an **artifact-only analysis** of the isolated SDF-to-smooth-boundary experiment. It reads previously generated CSV, JSON, and NPZ files; it never extracts a contour, fits a spline/Fourier curve, optimizes coefficients, or invokes a solver pipeline.

The three recorded methods are:

- **A:** periodic cubic-spline baseline;
- **B:** Fourier least-squares fit followed by arc-length reparameterization;
- **C:** SDF-constrained Fourier refinement followed by arc-length reparameterization and a short final refinement.

The analysis covers circle, ellipse, and star cases. It reports statuses and failures before numerical trends, keeps extraction-grid and representation-bandwidth axes separate, and does not choose a winner visually or collapse the metrics into one score. Set `SDF_BOUNDARY_ARTIFACT_DIR` to select a particular completed run.

In [ ]:
from pathlib import Path
import csv
import json
import math
import os
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

SHAPES = ("circle", "ellipse", "star")
METHODS = ("A", "B", "C")
DISPLAY_FLOOR = 1.0e-18

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "solvers").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root.")

REPO_ROOT = find_repo_root(Path.cwd())
configured = os.environ.get("SDF_BOUNDARY_ARTIFACT_DIR")
study_parent = REPO_ROOT / "results" / "sdf_boundary_parameterization"

def is_parameterization_bundle(path):
    manifest_path = path / "manifest.json"
    if not manifest_path.is_file() or not (path / "metrics.csv").is_file():
        return False
    with manifest_path.open(encoding="utf-8") as stream:
        manifest = json.load(stream)
    return manifest.get("measurement_scope") == "geometry_parameterization"

timestamped_roots = (
    sorted(
        (path for path in study_parent.iterdir() if path.is_dir() and is_parameterization_bundle(path)),
        key=lambda path: (path.stat().st_mtime, path.name),
        reverse=True,
    )
    if study_parent.is_dir()
    else []
)
default_roots = (*timestamped_roots, study_parent)
if configured:
    ARTIFACT_ROOT = Path(configured).expanduser().resolve()
else:
    populated = [root for root in default_roots if (root / "metrics.csv").is_file()]
    ARTIFACT_ROOT = populated[0] if populated else default_roots[0]

print("repository :", REPO_ROOT)
print("artifacts  :", ARTIFACT_ROOT)


## Load and validate the artifact bundle

`metrics.csv` is authoritative for row-wise analysis. `manifest.json` and `metrics.json` retain run-level configuration/provenance and nested records. NPZ files under `curves/` contain sampled arrays for inspection only. The loader tolerates the fallback names `summary.csv` and `results.csv`.

In [ ]:
def first_existing(root, names):
    for name in names:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return None

csv_path = first_existing(ARTIFACT_ROOT, ("metrics.csv", "summary.csv", "results.csv"))
manifest_path = first_existing(ARTIFACT_ROOT, ("manifest.json", "config.json"))
metrics_json_path = first_existing(ARTIFACT_ROOT, ("metrics.json", "summary.json"))

ROWS = []
if csv_path is not None:
    with csv_path.open(encoding="utf-8", newline="") as stream:
        ROWS = list(csv.DictReader(stream))
else:
    print("No metrics CSV found. Generate a comparison run or set SDF_BOUNDARY_ARTIFACT_DIR.")

MANIFEST = {}
if manifest_path is not None:
    with manifest_path.open(encoding="utf-8") as stream:
        MANIFEST = json.load(stream)

NESTED_METRICS = {}
if metrics_json_path is not None:
    with metrics_json_path.open(encoding="utf-8") as stream:
        NESTED_METRICS = json.load(stream)

curve_root = ARTIFACT_ROOT / "curves"
NPZ_PATHS = sorted(curve_root.glob("*.npz")) if curve_root.is_dir() else sorted(ARTIFACT_ROOT.glob("*.npz"))
print(f"CSV rows: {len(ROWS)}")
print(f"manifest keys: {sorted(MANIFEST) if isinstance(MANIFEST, dict) else 'non-mapping'}")
print(f"nested metrics loaded: {bool(NESTED_METRICS)}")
print(f"curve NPZ files: {len(NPZ_PATHS)}")


In [ ]:
def parse_scalar(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "n/a", "nan"}:
        return None
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        try:
            return float(text)
        except ValueError:
            return text

def pick(row, *names):
    for name in names:
        if name in row and row[name] not in (None, ""):
            return parse_scalar(row[name])
    for name in names:
        matches = [key for key in row if key.endswith("." + name)]
        if len(matches) == 1 and row[matches[0]] not in (None, ""):
            return parse_scalar(row[matches[0]])
    return None

def number(value):
    value = parse_scalar(value)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    if isinstance(value, str):
        token = value.lower().split("x", 1)[0].strip()
        try:
            return float(token)
        except ValueError:
            return None
    return None

def method_label(value):
    text = str(value or "").strip()
    aliases = {
        "a": "A", "method_a": "A", "periodic_spline": "A",
        "b": "B", "method_b": "B", "fourier_ls": "B",
        "c": "C", "method_c": "C", "fourier_sdf": "C",
    }
    return aliases.get(text.lower(), text.upper())

def shape_label(value):
    text = str(value or "").strip().lower()
    for canonical in ("circle", "ellipse", "star"):
        if canonical in text:
            return canonical
    return text

def normalized_record(row):
    return {
        "case_id": pick(row, "case_id", "run_id"),
        "shape": shape_label(pick(row, "shape", "case")),
        "grid": number(pick(row, "grid_resolution", "grid_shape", "grid")),
        "projected_samples": number(pick(row, "projected_samples", "sample_count")),
        "method": method_label(pick(row, "method", "method_name")),
        "bandwidth": number(pick(row, "bandwidth", "fourier_bandwidth")),
        "status": str(pick(row, "status") or "unknown").lower(),
        "failure_reason": pick(row, "failure_reason"),
        "frontend_id": pick(row, "frontend_id", "frontend_hash"),
        "sdf_max": number(pick(row, "metrics.sdf_residual.maximum_absolute", "sdf_residual.maximum_absolute", "maximum_absolute")),
        "sdf_rms": number(pick(row, "metrics.sdf_residual.rms", "sdf_residual.rms")),
        "normalized_sdf_max": number(pick(row, "metrics.sdf_residual.normalized_maximum", "sdf_residual.normalized_maximum")),
        "normalized_sdf_rms": number(pick(row, "metrics.sdf_residual.normalized_rms", "sdf_residual.normalized_rms")),
        "reference_hausdorff": number(pick(row, "metrics.reference_set.symmetric_hausdorff", "reference_set.symmetric_hausdorff")),
        "normal_max": number(pick(row, "metrics.reference_set.normal_angle_maximum_radians", "reference_set.normal_angle_maximum_radians")),
        "curvature_rms": number(pick(row, "metrics.reference_set.curvature_absolute_rms", "reference_set.curvature_absolute_rms")),
        "area_relative_error": number(pick(row, "metrics.integral_geometry.relative_area_error", "integral_geometry.relative_area_error")),
        "perimeter_relative_error": number(pick(row, "metrics.integral_geometry.relative_perimeter_error", "integral_geometry.relative_perimeter_error")),
        "area": number(pick(row, "metrics.integral_geometry.signed_area", "integral_geometry.signed_area", "area")),
        "perimeter": number(pick(row, "metrics.integral_geometry.perimeter", "integral_geometry.perimeter", "perimeter")),
        "minimum_speed": number(pick(row, "metrics.speed.minimum", "speed.minimum")),
        "speed_ratio": number(pick(row, "metrics.speed.ratio", "speed.ratio")),
        "self_intersections": number(pick(row, "metrics.topology.sampled_self_intersection_count", "topology.sampled_self_intersection_count")),
        "spectral_tail_0": number(pick(row, "metrics.spectral_tail.order_0", "spectral_tail.order_0")),
        "spectral_tail_1": number(pick(row, "metrics.spectral_tail.order_1", "spectral_tail.order_1")),
        "spectral_tail_2": number(pick(row, "metrics.spectral_tail.order_2", "spectral_tail.order_2")),
        "frozen_curve_sampling": pick(row, "metrics.frozen_curve_sampling", "frozen_curve_sampling") or [],
        "runtime_seconds": number(pick(row, "runtime_seconds", "timing.total_seconds", "total_runtime_seconds")),
        "raw": row,
    }

RECORDS = [normalized_record(row) for row in ROWS]
RECORDS = [record for record in RECORDS if record["shape"] or record["method"]]
print(f"normalized records: {len(RECORDS)}")


## Status, coverage, and fair-front-end audit

Failures and fallbacks remain evidence; they are never dropped from the status table. A successful numerical row is not automatically valid if it reports self-intersections or nonpositive speed. The front-end identifier audit checks whether A/B/C rows for one case/grid/sample configuration actually share the same extracted and projected loop.

In [ ]:
def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    prepared = [["" if item is None else str(item) for item in row] for row in rows]
    for row in prepared:
        widths = [max(width, len(item)) for width, item in zip(widths, row)]
    if not prepared:
        print("(no rows)")
        return
    print(" | ".join(str(header).ljust(width) for header, width in zip(headers, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in prepared:
        print(" | ".join(item.ljust(width) for item, width in zip(row, widths)))

status_counts = Counter((record["shape"], record["method"], record["status"]) for record in RECORDS)
status_rows = []
for shape in SHAPES:
    for method in METHODS:
        statuses = {status: count for (item_shape, item_method, status), count in status_counts.items() if item_shape == shape and item_method == method}
        status_rows.append((shape, method, statuses.get("success", 0), statuses.get("fallback", 0), statuses.get("failed", 0), sum(statuses.values())))
print_table(("shape", "method", "success", "fallback", "failed", "total"), status_rows)

problem_rows = [record for record in RECORDS if record["status"] != "success" or (record["self_intersections"] or 0) > 0 or (record["minimum_speed"] is not None and record["minimum_speed"] <= 0)]
print("\nFailure/fallback/invalid rows:")
print_table(
    ("case", "shape", "method", "grid", "K", "status", "reason"),
    [(item["case_id"], item["shape"], item["method"], item["grid"], item["bandwidth"], item["status"], item["failure_reason"]) for item in problem_rows],
)

frontends = defaultdict(list)
for record in RECORDS:
    key = (record["shape"], record["grid"], record["projected_samples"])
    frontends[key].append(record["frontend_id"])
mismatched = []
for key, identifiers in frontends.items():
    present = {str(identifier) for identifier in identifiers if identifier is not None}
    missing = sum(identifier is None for identifier in identifiers)
    if len(present) != 1 or missing:
        mismatched.append((key, sorted(present), missing))
print(f"\nshared-front-end identifier mismatches: {len(mismatched)}")
for key, values, missing in mismatched[:10]:
    print(key, values, f"missing={missing}")
assert not mismatched, "A/B/C rows do not all share exactly one front-end identifier per shape/grid/sample case."


## Required metrics table

This compact view preserves the independent fidelity, regularity, topology, spectrum, and runtime diagnostics. It is intentionally not an aggregate ranking.

In [ ]:
def compact(value, digits=3):
    if value is None:
        return "--"
    if isinstance(value, float):
        return f"{value:.{digits}e}"
    return str(value)

ordered = sorted(RECORDS, key=lambda item: (item["shape"], item["grid"] or -1, item["method"], item["bandwidth"] or -1))
metric_rows = [
    (
        item["shape"], item["method"], compact(item["grid"], 0), compact(item["projected_samples"], 0), compact(item["bandwidth"], 0), item["status"],
        compact(item["sdf_max"]), compact(item["sdf_rms"]), compact(item["normalized_sdf_max"]), compact(item["normalized_sdf_rms"]),
        compact(item["reference_hausdorff"]), compact(item["normal_max"]),
        compact(item["minimum_speed"]), compact(item["speed_ratio"]), compact(item["self_intersections"], 0),
        compact(item["area"]), compact(item["perimeter"]), compact(item["area_relative_error"]), compact(item["perimeter_relative_error"]),
        compact(item["spectral_tail_0"]), compact(item["spectral_tail_1"]), compact(item["spectral_tail_2"]), compact(item["runtime_seconds"]),
    )
    for item in ordered
]
print_table(
    ("shape", "method", "grid", "M", "K", "status", "F max", "F RMS", "norm F max", "norm F RMS", "set H", "normal max", "min speed", "speed ratio", "self-X", "area", "perimeter", "area rel", "perim rel", "tail d0", "tail d1", "tail d2", "time s"),
    metric_rows,
)


## Extraction-grid convergence at fixed representation settings

Grid refinement and representation refinement answer different questions. Each line below fixes method, projected sample count, and (for B/C) bandwidth, then varies only the Cartesian grid. Exact Newton projection on analytic fields may produce an early plateau; a plateau is reported rather than treated as failure or hidden by selecting another axis. Failed/fallback rows are excluded from convergence lines but remain in the status table above.

In [ ]:
def chosen_error(record):
    for name in ("reference_hausdorff", "normalized_sdf_max", "sdf_max"):
        value = record.get(name)
        if value is not None and math.isfinite(value):
            return abs(float(value)), name
    return None, None

grid_groups = defaultdict(list)
for record in RECORDS:
    error, error_name = chosen_error(record)
    if record["status"] == "success" and record["grid"] and error is not None:
        key = (record["shape"], record["method"], record["bandwidth"], record["projected_samples"], error_name)
        grid_groups[key].append((record["grid"], error))

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8), sharey=False)
for ax, shape in zip(axes, SHAPES):
    plotted = 0
    for (item_shape, method, bandwidth, sample_count, error_name), pairs in sorted(grid_groups.items(), key=str):
        unique = sorted(set(pairs))
        if item_shape != shape or len(unique) < 2:
            continue
        grids = np.asarray([pair[0] for pair in unique], dtype=float)
        errors = np.asarray([max(pair[1], DISPLAY_FLOOR) for pair in unique])
        spacing = 1.0 / np.maximum(grids - 1.0, 1.0)
        label = method if bandwidth is None else f"{method}, K={bandwidth:g}"
        ax.loglog(spacing, errors, marker="o", linewidth=1.0, markersize=3, label=label)
        plotted += 1
    ax.set_title(shape)
    ax.set_xlabel("grid-spacing proxy 1/(n-1)")
    ax.set_ylabel("set / normalized-field error")
    ax.grid(True, which="both", alpha=0.25)
    ax.invert_xaxis()
    if plotted:
        ax.legend(fontsize=6)
    else:
        ax.text(0.5, 0.5, "need >=2 grids", transform=ax.transAxes, ha="center")
fig.suptitle("Grid convergence with representation settings fixed")
fig.tight_layout()
plt.show()

rate_rows = []
for key, pairs in sorted(grid_groups.items(), key=str):
    unique = sorted(set(pairs))
    for (coarse_grid, coarse_error), (fine_grid, fine_error) in zip(unique, unique[1:]):
        if coarse_error > 0.0 and fine_error > 0.0 and fine_grid != coarse_grid:
            rate = math.log(coarse_error / fine_error) / math.log((fine_grid - 1.0) / (coarse_grid - 1.0))
        else:
            rate = None
        trend = "decrease" if fine_error < coarse_error / 1.05 else ("increase" if fine_error > coarse_error * 1.05 else "plateau")
        shape, method, bandwidth, sample_count, error_name = key
        rate_rows.append((shape, method, bandwidth, int(coarse_grid), int(fine_grid), error_name, compact(rate), trend))
print_table(("shape", "method", "K", "grid 1", "grid 2", "metric", "observed p", "trend"), rate_rows)


## Fourier-bandwidth convergence at fixed extraction grid

Only Methods B and C have a Fourier bandwidth. For each shape and method, the plot uses successful rows on the finest available grid while holding projected sample count fixed. Method A belongs in a separate projected-sample/knot-count study and is not duplicated across artificial K values. Nonmonotone steps are retained: arc-length refitting and derivative amplification can create plateaus or local regressions.

In [ ]:
finest_grid = {}
for shape in SHAPES:
    available = [record["grid"] for record in RECORDS if record["shape"] == shape and record["grid"]]
    finest_grid[shape] = max(available) if available else None

bandwidth_groups = defaultdict(list)
for record in RECORDS:
    error, error_name = chosen_error(record)
    if (
        record["method"] in {"B", "C"}
        and record["status"] == "success"
        and record["grid"] == finest_grid.get(record["shape"])
        and record["bandwidth"] is not None
        and error is not None
    ):
        key = (record["shape"], record["method"], record["grid"], record["projected_samples"], error_name)
        bandwidth_groups[key].append((record["bandwidth"], error))

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8), sharey=False)
for ax, shape in zip(axes, SHAPES):
    plotted = 0
    for (item_shape, method, grid, sample_count, error_name), pairs in sorted(bandwidth_groups.items(), key=str):
        unique = sorted(set(pairs))
        if item_shape != shape or len(unique) < 2:
            continue
        bandwidths = np.asarray([pair[0] for pair in unique], dtype=float)
        errors = np.asarray([max(pair[1], DISPLAY_FLOOR) for pair in unique])
        label = f"{method}, M={sample_count:g}" if sample_count is not None else method
        ax.semilogy(bandwidths, errors, marker="o", linewidth=1.0, markersize=3, label=label)
        plotted += 1
    ax.set_title(f"{shape}, grid={finest_grid.get(shape)}")
    ax.set_xlabel("Fourier bandwidth K")
    ax.set_ylabel("set / normalized-field error")
    ax.grid(True, which="both", alpha=0.25)
    if plotted:
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "need >=2 bandwidths", transform=ax.transAxes, ha="center")
fig.suptitle("Bandwidth convergence on each shape's finest extraction grid")
fig.tight_layout()
plt.show()

bandwidth_rows = []
for key, pairs in sorted(bandwidth_groups.items(), key=str):
    unique = sorted(set(pairs))
    for (low_k, low_error), (high_k, high_error) in zip(unique, unique[1:]):
        ratio = high_error / low_error if low_error > 0.0 else None
        trend = "decrease" if high_error < low_error / 1.05 else ("increase" if high_error > low_error * 1.05 else "plateau")
        shape, method, grid, sample_count, error_name = key
        bandwidth_rows.append((shape, method, int(grid), int(low_k), int(high_k), error_name, compact(ratio), trend))
print_table(("shape", "method", "grid", "K 1", "K 2", "metric", "error ratio", "trend"), bandwidth_rows)


## Frozen-curve even-node convergence

The final node count $N$ is independent of extraction grid, projected sample count $M$, and Fourier bandwidth $K$. Each row below samples the same accepted continuous curve at the configured sequence of even node counts without refitting it. The periodic $ds$-weight sum should converge to the dense reference perimeter while the sampled geometry remains finite, regular, counterclockwise, and endpoint-free.

In [ ]:
frozen_rows = []
for record in ordered:
    samples = record["frozen_curve_sampling"]
    if not isinstance(samples, list):
        continue
    samples = sorted((item for item in samples if isinstance(item, dict)), key=lambda item: item.get("num_nodes", -1))
    for lower, upper in zip(samples, samples[1:]):
        low_error = number(lower.get("perimeter_absolute_error"))
        high_error = number(upper.get("perimeter_absolute_error"))
        ratio = high_error / low_error if low_error not in (None, 0.0) and high_error is not None else None
        ready = all(bool(upper.get(name)) for name in ("all_finite", "positive_speed", "counterclockwise")) and not bool(upper.get("includes_repeated_endpoint"))
        frozen_rows.append((record["shape"], record["method"], compact(record["grid"], 0), compact(record["projected_samples"], 0), compact(record["bandwidth"], 0), lower.get("num_nodes"), upper.get("num_nodes"), compact(low_error), compact(high_error), compact(ratio), "ready" if ready else "invalid"))
print_table(("shape", "method", "grid", "M", "K", "N 1", "N 2", "perim err 1", "perim err 2", "error ratio", "geometry"), frozen_rows)

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8), sharey=False)
for ax, shape in zip(axes, SHAPES):
    shape_records = [record for record in RECORDS if record["shape"] == shape and isinstance(record["frozen_curve_sampling"], list) and record["frozen_curve_sampling"]]
    if shape_records:
        finest = max(record["grid"] for record in shape_records if record["grid"] is not None)
        most_samples = max(record["projected_samples"] for record in shape_records if record["grid"] == finest and record["projected_samples"] is not None)
        shape_records = [record for record in shape_records if record["grid"] == finest and record["projected_samples"] == most_samples]
    for method in METHODS:
        candidates = [record for record in shape_records if record["method"] == method]
        if not candidates:
            continue
        record = max(candidates, key=lambda item: item["bandwidth"] or -1)
        samples = sorted(record["frozen_curve_sampling"], key=lambda item: item["num_nodes"])
        node_counts = [item["num_nodes"] for item in samples]
        errors = [max(float(item["perimeter_absolute_error"]), DISPLAY_FLOOR) for item in samples]
        label = method if method == "A" else f"{method}, K={int(record['bandwidth'])}"
        ax.loglog(node_counts, errors, marker="o", label=label)
    ax.set_title(shape)
    ax.set_xlabel("even frozen-curve nodes N")
    ax.set_ylabel("|sum(ds weights) - dense perimeter|")
    ax.grid(True, which="both", alpha=0.25)
    if ax.lines:
        ax.legend(fontsize=7)
fig.suptitle("Frozen continuous curves sampled at multiple even node counts (no refit)")
fig.tight_layout()
plt.show()


## Inspect stored curve arrays (no recomputation)

The NPZ inventory confirms which raw, projected, fitted, and reference samples were retained. The optional gallery plots stored arrays only; it does not reconstruct a parameterization or rerun a method.

In [ ]:
inventory = []
for path in NPZ_PATHS:
    with np.load(path, allow_pickle=False) as archive:
        description = "; ".join(f"{name}:{archive[name].shape}/{archive[name].dtype}" for name in archive.files)
    inventory.append((path.name, description))
print_table(("file", "arrays (shape/dtype)"), inventory[:30])
if len(inventory) > 30:
    print(f"... {len(inventory) - 30} more files")

def choose_array(archive, names):
    for name in names:
        if name in archive.files:
            values = np.asarray(archive[name])
            if values.ndim == 2 and values.shape[1] == 2:
                return values
    return None

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.8))
for ax, shape in zip(axes, SHAPES):
    candidates = [path for path in NPZ_PATHS if shape in path.stem.lower()]
    if not candidates:
        ax.text(0.5, 0.5, "no stored NPZ", transform=ax.transAxes, ha="center")
        ax.set_title(shape)
        continue
    path = candidates[0]
    with np.load(path, allow_pickle=False) as archive:
        raw = choose_array(archive, ("raw_contour", "raw_points"))
        projected = choose_array(archive, ("projected_points", "projected_contour"))
        fitted = choose_array(archive, ("fitted_points", "curve_points", "dense_points", "points"))
        reference = choose_array(archive, ("reference_points", "target_points"))
    for values, style, label in ((raw, ".-", "raw"), (projected, ".", "projected"), (fitted, "-", "fitted"), (reference, "--", "reference")):
        if values is not None:
            closed = np.vstack((values, values[0])) if style != "." else values
            ax.plot(closed[:, 0], closed[:, 1], style, linewidth=1.0, markersize=2.0, label=label)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{shape}: {path.name}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7)
fig.suptitle("Representative stored arrays (first NPZ per shape)")
fig.tight_layout()
plt.show()


## Evidence checklist, not a winner selection

A defensible comparison asks whether each method remains valid, whether error decreases or reaches a documented floor as the grid and bandwidth are refined independently, and which stage limits convergence. A low SDF residual cannot compensate for self-intersection, vanishing speed, missing components, or a fallback. Plots are diagnostic views; conclusions should cite the status and convergence tables above.

In [ ]:
check_rows = []
for shape in SHAPES:
    subset = [record for record in RECORDS if record["shape"] == shape]
    successful_methods = sorted({record["method"] for record in subset if record["status"] == "success"})
    grids = sorted({int(record["grid"]) for record in subset if record["grid"] is not None})
    bandwidths = sorted({int(record["bandwidth"]) for record in subset if record["bandwidth"] is not None})
    failures = sum(record["status"] != "success" for record in subset)
    invalid = sum((record["self_intersections"] or 0) > 0 or (record["minimum_speed"] is not None and record["minimum_speed"] <= 0) for record in subset)
    check_rows.append((shape, successful_methods, grids, bandwidths, failures, invalid))
print_table(("shape", "successful methods", "grids", "bandwidths", "failed/fallback", "invalid geometry"), check_rows)
print("\nInterpret plateaus and regressions from the two convergence tables; do not infer a winner from line color or a single scalar.")
